In [1]:
#acessar probabilidade de chuva nos bairros das estações

import requests
from datetime import datetime

# Substitua pela sua chave que você pegou no painel da OpenWeatherMap
API_KEY = "4bb8a51d91d36972b6e508359c61ef2f"

# O seu dicionário com as coordenadas exatas
bairros_recife = {
    "Campina do Barreto": {"lat": -8.013000, "lon": -34.881000},
    "Torreão": {"lat": -8.037000, "lon": -34.884000},
    "RECIFE - APAC": {"lat": -8.044910, "lon": -34.875180},
    "Imbiribeira": {"lat": -8.120975, "lon": -34.913983},
    "Dois Irmãos": {"lat": -8.018378, "lon": -34.947058}
}

def consultar_chuva_bairro(nome_bairro):
    bairro = bairros_recife.get(nome_bairro)
    
    if not bairro:
        print(f"Bairro '{nome_bairro}' não encontrado no dicionário.")
        return
    
    lat = bairro["lat"]
    lon = bairro["lon"]
    
    # Chamada para a API de Previsão de 5 dias / 3 horas
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric&lang=pt_br"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados = response.json()
        print(f"\n=== Previsão de Chuva para: {nome_bairro} ===")
        
        # Pegamos a data de hoje no formato 'AAAA-MM-DD' para filtrar os resultados
        hoje = datetime.now().strftime('%Y-%m-%d')
        encontrou_previsao = False
        
        for previsao in dados['list']:
            data_hora_texto = previsao['dt_txt'] # Exemplo: "2026-06-16 12:00:00"
            
            # Se a previsão pertence ao dia de hoje, nós exibimos
            if data_hora_texto.startswith(hoje):
                encontrou_previsao = True
                
                # Extrai apenas o horário (HH:MM) para ficar mais limpo
                horario = data_hora_texto.split(" ")[1][:5]
                
                # O 'pop' da OpenWeather vem de 0.0 a 1.0 (multiplicamos por 100 para virar %)
                probabilidade = previsao.get('pop', 0) * 100
                descricao = previsao['weather'][0]['description']
                
                # Opcional: Volume de chuva estimado nas últimas 3 horas se houver acumulado
                volume_chuva = previsao.get('rain', {}).get('3h', 0)
                txt_volume = f" | Vol. Estimado: {volume_chuva}mm" if volume_chuva > 0 else ""
                
                print(f"Horário: {horario} -> Chance de Chuva: {probabilidade:.0f}% ({descricao}){txt_volume}")
        
        if not encontrou_previsao:
            print("Não há mais previsões disponíveis para o dia de hoje. Tente consultar as próximas datas mudando o filtro do código.")
                
    elif response.status_code == 401:
        print("Erro 401: Chave de API inválida ou ainda não ativada. Lembre-se que novas chaves podem levar algumas horas para funcionar.")
    else:
        print(f"Erro na requisição. Status: {response.status_code}")

# --- Exemplos de Uso ---
# Você pode chamar para um bairro específico:
consultar_chuva_bairro("RECIFE - APAC")
consultar_chuva_bairro("Imbiribeira")
consultar_chuva_bairro("Dois Irmãos")
consultar_chuva_bairro("Campina do Barreto")
consultar_chuva_bairro("Torreão")

# Ou rodar um loop para ver o panorama de todos de uma vez só:
# print("\n--- PANORAMA GERAL DO RECIFE ---")
# for bairro in bairros_recife.keys():
#     consultar_chuva_bairro(bairro)


=== Previsão de Chuva para: RECIFE - APAC ===
Horário: 12:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 4.59mm
Horário: 15:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 3.11mm
Horário: 18:00 -> Chance de Chuva: 100% (chuva leve) | Vol. Estimado: 2.28mm
Horário: 21:00 -> Chance de Chuva: 80% (chuva leve) | Vol. Estimado: 0.5mm

=== Previsão de Chuva para: Imbiribeira ===
Horário: 12:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 4.43mm
Horário: 15:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 3.03mm
Horário: 18:00 -> Chance de Chuva: 100% (chuva leve) | Vol. Estimado: 2.38mm
Horário: 21:00 -> Chance de Chuva: 84% (chuva leve) | Vol. Estimado: 0.56mm

=== Previsão de Chuva para: Dois Irmãos ===
Horário: 12:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 5.06mm
Horário: 15:00 -> Chance de Chuva: 100% (chuva moderada) | Vol. Estimado: 3.53mm
Horário: 18:00 -> Chance de Chuva: 100% (chuva leve) | Vol. Estimado: 2.

Mapa meteorológico

In [7]:
pip install folium requests ipython

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import folium
from IPython.display import display

# Sua chave da OpenWeatherMap
API_KEY = "4bb8a51d91d36972b6e508359c61ef2f"

# Dicionário com as coordenadas exatas das estações/bairros
bairros_recife = {
    "Campina do Barreto": {"lat": -8.013000, "lon": -34.881000},
    "Torreão": {"lat": -8.037000, "lon": -34.884000},
    "RECIFE - APAC": {"lat": -8.044910, "lon": -34.875180},
    "Imbiribeira": {"lat": -8.120975, "lon": -34.913983},
    "Dois Irmãos": {"lat": -8.018378, "lon": -34.947058}
}

# 1. Criar o mapa base centralizado em Recife (com zoom adequado para os bairros)
mapa = folium.Map(location=[-8.05428, -34.92842], zoom_start=12, tiles="OpenStreetMap")

# 2. Adicionar os marcadores (alfinetes) de cada bairro no mapa
for nome, coord in bairros_recife.items():
    folium.Marker(
        location=[coord["lat"], coord["lon"]],
        popup=nome,
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(mapa)

# 3. Configurar a camada (layer) de radar meteorológico da OpenWeather
layer = "precipitation_new"
url_tile_openweather = f"https://tile.openweathermap.org/map/{layer}/{{z}}/{{x}}/{{y}}.png?appid={API_KEY}"

# 4. Adicionar a sobreposição das manchas de chuva (TileLayer) no mapa
folium.TileLayer(
    tiles=url_tile_openweather,
    attr="OpenWeatherMap",
    name="Precipitação (Chuva)",
    overlay=True,
    control=True,
    opacity=0.7 # Define 70% de opacidade para conseguir ver as ruas por baixo da chuva
).add_to(mapa)

# 5. Comando específico para renderizar o mapa na saída de célula do VS Code
# Substitua a última linha do seu código por isto:
from IPython.display import IFrame

# 1. Salva o mapa temporariamente em um arquivo oculto local
mapa.save("temp_mapa_chuva.html")

# 2. Força o VS Code a ler o arquivo através de um componente IFrame controlado
# O argumento width="100%" e height=500 define o tamanho do bloco que aparecerá no notebook
display(IFrame(src="temp_mapa_chuva.html", width="100%", height=500))

In [12]:
import webbrowser
import folium

API_KEY = "4bb8a51d91d36972b6e508359c61ef2f"

bairros_recife = {
    "Campina do Barreto": {"lat": -8.013000, "lon": -34.881000},
    "Torreão": {"lat": -8.037000, "lon": -34.884000},
    "RECIFE - APAC": {"lat": -8.044910, "lon": -34.875180},
    "Imbiribeira": {"lat": -8.120975, "lon": -34.913983},
    "Dois Irmãos": {"lat": -8.018378, "lon": -34.947058}
}

# Afastamos ligeiramente o zoom inicial para 10 para garantir que apanha o radar completo
mapa = folium.Map(location=[-8.05428, -34.92842], zoom_start=10, tiles="OpenStreetMap")

for nome, coord in bairros_recife.items():
    folium.Marker(
        location=[coord["lat"], coord["lon"]],
        popup=nome,
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(mapa)

layer = "precipitation_new"
url_tile_openweather = f"https://tile.openweathermap.org/map/{layer}/{{z}}/{{x}}/{{y}}.png?appid={API_KEY}"

# Criamos a camada de chuva com um nome claro
camada_chuva = folium.TileLayer(
    tiles=url_tile_openweather,
    attr="OpenWeatherMap",
    name="Radar de Chuva (OpenWeather)",
    overlay=True,
    control=True,
    opacity=0.7
)
camada_chuva.add_to(mapa)

# --- O AJUSTE ESTÁ AQUI: Adiciona o botão de controle no canto do mapa ---
folium.LayerControl(position="topright").add_to(mapa)

nome_arquivo = "mapa_radar_chuva.html"
mapa.save(nome_arquivo)
webbrowser.open(nome_arquivo)

print(f"✅ Mapa atualizado com controle de camadas aberto no navegador!")

✅ Mapa atualizado com controle de camadas aberto no navegador!


In [21]:
import webbrowser
import folium

API_KEY = "4bb8a51d91d36972b6e508359c61ef2f"

bairros_recife = {
    "Campina do Barreto": {"lat": -8.013000, "lon": -34.881000},
    "Torreão": {"lat": -8.037000, "lon": -34.884000},
    "RECIFE - APAC": {"lat": -8.044910, "lon": -34.875180},
    "Imbiribeira": {"lat": -8.120975, "lon": -34.913983},
    "Dois Irmãos": {"lat": -8.018378, "lon": -34.947058}
}

# 1. Criamos o mapa com um fundo totalmente preto e limpo (sem ruas) para receber a chuva
mapa = folium.Map(location=[-8.05428, -34.92842], zoom_start=11, tiles="CartoDB DarkMatter")

# 2. Adicionamos a camada de CHUVA primeiro (ela vai ficar por baixo das linhas das ruas)
url_chuva = f"https://tile.openweathermap.org/map/precipitation_new/{{z}}/{{x}}/{{y}}.png?appid={API_KEY}"
camada_chuva = folium.TileLayer(
    tiles=url_chuva,
    attr="OpenWeatherMap",
    name="🌧️ Radar de Chuva (Neon)",
    overlay=True,
    control=True,
    opacity=1.0,
    show=True
)
camada_chuva.add_to(mapa)

# 3. Adicionamos a camada de VENTO
url_vento = f"https://tile.openweathermap.org/map/wind_new/{{z}}/{{x}}/{{y}}.png?appid={API_KEY}"
camada_vento = folium.TileLayer(
    tiles=url_vento,
    attr="OpenWeatherMap",
    name="💨 Intensidade do Vento",
    overlay=True,
    control=True,
    opacity=1.0,
    show=False
)
camada_vento.add_to(mapa)

# 4. TRUQUE CORRIGIDO: Aplicamos o filtro NEON apenas nas imagens que contêm "openweathermap" na URL.
# Isso protege o mapa de fundo e as linhas do CartoDB de serem distorcidos!
mapa.get_root().header.add_child(folium.Element("""
    <style>
        img[src*="openweathermap.org"] {
            filter: saturate(3.5) contrast(1.8) brightness(1.3) !important;
        }
    </style>
"""))

# 5. Adicionamos apenas as LINHAS e DELIMITAÇÕES por cima de tudo (CartoDB DarkMatter Only Lines)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/dark_all_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; CartoDB",
    name="📍 Ruas e Delimitações",
    overlay=True,
    control=False, # Fica sempre visível para guiar o usuário
    opacity=1.0
).add_to(mapa)

# 6. Por fim, colocamos os marcadores dos bairros no topo de todas as camadas
for nome, coord in bairros_recife.items():
    folium.Marker(
        location=[coord["lat"], coord["lon"]],
        popup=nome,
        icon=folium.Icon(color="orange", icon="info-sign")
    ).add_to(mapa)

# Menu de controle no canto superior direito
folium.LayerControl(position="topright").add_to(mapa)

# Configuração para atualizar a página sozinho a cada 5 minutos
mapa.get_root().header.add_child(folium.Element('<meta http-equiv="refresh" content="300">'))

# Salvar e abrir no navegador
nome_arquivo = "mapa_meteorologico_perfeito.html"
mapa.save(nome_arquivo)
webbrowser.open(nome_arquivo)

print(f"✅ Mapa corrigido! Agora as ruas aparecem nitidamente por cima do efeito neon.")

✅ Mapa corrigido! Agora as ruas aparecem nitidamente por cima do efeito neon.


In [3]:
import webbrowser
import folium

API_KEY = "4bb8a51d91d36972b6e508359c61ef2f"

# 1. Mapa Base Escuro em visão macro (Brasil)
mapa = folium.Map(location=[-14.2350, -51.9253], zoom_start=4, min_zoom=3, max_zoom=7, tiles="CartoDB DarkMatter")

# 2. Camada de chuva estável da OpenWeather
url_chuva = f"https://tile.openweathermap.org/map/precipitation_new/{{z}}/{{x}}/{{y}}.png?appid={API_KEY}"

camada_chuva = folium.TileLayer(
    tiles=url_chuva,
    attr="OpenWeatherMap",
    name="🌧️ Radar Estilo iOS",
    overlay=True,
    control=True,
    opacity=0.9,
    show=True
)
camada_chuva.add_to(mapa)

# 3. FILTRO DE DESIGN DESIGN ESTILO APPLE (Efeito Neon Fosco / Glassmorphism)
# Misturamos desfoque sutil, saturação extrema e alteração de matriz de cor.
mapa.get_root().header.add_child(folium.Element("""
    <style>
        img[src*="openweathermap.org"] {
            filter: hue-rotate(160deg) saturate(4.5) brightness(1.3) contrast(1.2) blur(0.4px) !important;
            mix-blend-mode: screen;
        }
    </style>
"""))

# 4. Linhas dos estados e nomes por cima das manchas (como no iPhone)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/dark_all_labels/{z}/{x}/{y}{r}.png",
    attr="&copy; CartoDB",
    overlay=True,
    control=False,
    opacity=0.8
).add_to(mapa)

# Menu de controle
folium.LayerControl(position="topright").add_to(mapa)

# Salvar e abrir
nome_arquivo = "radar_estilo_apple.html"
mapa.save(nome_arquivo)
webbrowser.open(nome_arquivo)

print("🍏 Mapa gerado com a estética visual inspirada no radar do iPhone!")

🍏 Mapa gerado com a estética visual inspirada no radar do iPhone!


Uso da biblioteca Cartopy para chuva via radar

In [4]:
!pip install cartopy matplotlib requests


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
!pip install plotly requests pandas


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd
import plotly.express as px
import requests

# 1. Configurar as coordenadas de Recife
lat_centro, lon_centro = -8.05428, -34.8813

# 2. Buscar o histórico de radar da API do RainViewer
url_api = "https://api.rainviewer.com/public/weather-maps.json"
resposta = requests.get(url_api).json()
lista_passada = resposta.get("radar", {}).get("past", [])

if not lista_passada:
    print("Não foi possível obter o histórico do radar.")
else:
    # 3. Criar o DataFrame base para os frames da animação
    dados_mapa = []
    for idx, item in enumerate(lista_passada):
        dados_mapa.append({
            "Frame": f"Tempo {idx + 1}",
            "Latitude": lat_centro,
            "Longitude": lon_centro,
            "Cidade": "Recife"
        })
    df = pd.DataFrame(dados_mapa)

    # 4. Inicializar o mapa base (CORREÇÃO: Travando o zoom em 6, que é inteiro)
    zoom_valido = 6
    fig = px.scatter_mapbox(
        df,
        lat="Latitude",
        lon="Longitude",
        text="Cidade",
        animation_frame="Frame",
        zoom=zoom_valido,
        height=700
    )

    # URLs dos servidores de mapas
    url_fundo_escuro = "https://basemaps.cartocdn.com/dark_all/{z}/{x}/{y}.png"
    codigo_cor = 2  # Paleta estilo NOAA

    # Primeiro frame de radar
    primeiro_path = lista_passada[0]["path"]
    url_primeiro_radar = f"https://tilecache.rainviewer.com{primeiro_path}/256/{{z}}/{{x}}/{{y}}/{codigo_cor}/1_1.png"

    # 5. Configurar o layout inicial do mapa (Modo Escuro)
    fig.update_layout(
        mapbox_style="white-bg",
        mapbox=dict(
            center=dict(lat=lat_centro, lon=lon_centro),
            zoom=zoom_valido,
            layers=[
                # Camada 0: Mapa urbano escuro fixo
                {
                    "below": "traces",
                    "sourcetype": "raster",
                    "source": [url_fundo_escuro]
                },
                # Camada 1: Radar de chuva do instante inicial
                {
                    "below": "traces",
                    "sourcetype": "raster",
                    "source": [url_primeiro_radar],
                    "opacity": 0.75
                }
            ]
        ),
        paper_bgcolor="#11151c",
        margin={"r": 0, "t": 40, "l": 0, "b": 0},
        title={
            "text": "Radar Meteorológico Ao Vivo - Recife",
            "font": {"color": "white", "size": 16},
            "y": 0.98,
            "x": 0.5,
            "xanchor": "center"
        }
    )

    # 6. Atualizar as camadas de cada frame individual
    for frame, dados_frame in zip(fig.frames, lista_passada):
        path_radar = dados_frame["path"]
        url_radar_frame = f"https://tilecache.rainviewer.com{path_radar}/256/{{z}}/{{x}}/{{y}}/{codigo_cor}/1_1.png"
        
        frame["layout"] = {
            "mapbox": {
                "center": {"lat": lat_centro, "lon": lon_centro},
                "zoom": zoom_valido, # Garante o mesmo zoom inteiro em todos os passos
                "layers": [
                    # Mantém o fundo escuro ativo
                    {
                        "below": "traces",
                        "sourcetype": "raster",
                        "source": [url_fundo_escuro]
                    },
                    # Altera para a mancha do tempo atual
                    {
                        "below": "traces",
                        "sourcetype": "raster",
                        "source": [url_radar_frame],
                        "opacity": 0.75
                    }
                ]
            }
        }

    # Estilizar o marcador vermelho de Recife
    fig.update_traces(
        marker=dict(size=12, color="#ff3333"),
        textposition="top right",
        textfont=dict(color="white", size=12, family="Arial Black")
    )

    # Exibe o mapa atualizado
    fig.show()

C:\Users\windows 10\AppData\Local\Temp\ipykernel_14292\3740757067.py:29: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [24]:
# Salva o arquivo HTML corrigido na sua pasta
fig.write_html("radar_animado_recife.html")